# Week 6 - Logistic Regression

**Student Name 1, Student Name 2, Student Name 3**

### Aims

The main concepts covered in this notebook are:

>* logistic regression
>* performance metrics for classification
>* dealing with imbalanced data 
>* multi-class logistic regression

1. [Setup](#setup)

2. [Binary Logistic Regression](#RBH)

3. [Regularisation](#SKV)

4. [Imbalanced Data](#Imbal)

5. [Multi-class Example](#mclog)


This week we will be implementing logistic regression for a classification task. 

- We will mainly focus on the data set stored in `Default.csv`.
- For the multi-class example, we consider the `iris data` at the end of the notebook.

You may complete the worksheets together in teams of 2-3. When completing worksheets:

>- You will have tasks tagged by (CORE) and (EXTRA). 
>- Your primary aim is to complete the (CORE) components during the WS session, afterwards you can try to complete the (EXTRA) tasks for your self-learning process. 
>- In some Exercises, you will see some beneficial hints at the bottom of questions.

Instructions for submitting your workshops can be found at the end of worksheet. As a reminder, you must submit a pdf of your notebook on Learn by 16:00 PM on the first Monday after the day workshop was given.

---

# Setup <a id='setup'></a>

## Packages

Let's load the packages we need for this workshop.

In [ ]:
# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn
from sklearn.linear_model import LogisticRegression

# Other necessary packages
from sklearn.preprocessing import StandardScaler # scaling features
from sklearn.pipeline import make_pipeline           # combining classifier steps
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold 
from sklearn.model_selection import train_test_split

## Helper Functions

We will make use of one of the helper functions that we saw last week.

In [ ]:
def get_coefs(m, plot = False, feature_names = None, figsize = (5,5), figtitle = None, intercept = True):
    """Returns model coefficients in a data frame for a fitted linear model.
    
    Args:
        m: sklearn linear model object or pipeline with linear model as final step
        plot: boolean value, should coefficients be plotted with error bars
        feature_names: list of feature names to use in the plot 
        figsize: tuple defining figure size
        figtitle: string defining figure title
        intercept: boolean value, should intercept be included in the plot
    """
    
    # Extract intercept and coefficients into a single array
    w0 = m[-1].intercept_ if isinstance(m, sklearn.pipeline.Pipeline) else m.intercept_
    w1 = m[-1].coef_ if isinstance(m, sklearn.pipeline.Pipeline) else m.coef_
    w = np.concatenate((w0.reshape(-1), w1.reshape(-1)))
    # Extract name of features
    if feature_names is None:
        feature_names = m[:-1].get_feature_names_out() if isinstance(m, sklearn.pipeline.Pipeline) else m.feature_names_in_
    feature_names = np.concatenate((['intercept'], feature_names))
    # Create a data frame
    w_df = pd.DataFrame({'feature': feature_names, 'coef': w}).sort_values ("coef", ascending=False)

    if plot:
        if not intercept:
            w_df = w_df[w_df['feature'] != 'intercept']
        plt.figure(figsize=figsize)
        plt.barh(w_df['feature'], w_df['coef'])
        plt.ylabel('Features')
        plt.xlabel('Coefficient Value')
        plt.axvline(x=0, color=".5")
        if figtitle is not None:
            plt.title(figtitle)
        plt.grid()
        plt.show()
    
    return  w_df

## Data

The dataset collects information on **10000** individuals, recording whether they defaulted on their credit card or not as well as other characteristics. Specifically, the included columns in the data are:

* `default` - Whether the individual has defaulted

* `student` - Whether the individual is a student

* `balance` - The balance in the individual's account

* `income` - Income of an individual

Our aim is to build a model using Logistic Regression to predict if person will default or not.

In [ ]:
df_default = pd.read_csv("Default.csv", index_col=0)

df_default.head()

## Exploratory Data Analysis

We will start with explanatory data analysis (EDA) to get more insight about the data. 

### 🚩 Exercise 1 (CORE)

Examine the structure of the data. Consider the following questionss: What are the types of each variable? Based on the descriptive statistics, do you anticipate any feature engineering steps that may be needed? Are there any missing values in the data? 

In [ ]:
# Info on data types 
df_default.info()

In [ ]:
# Descriptive statistics
print(df_default.describe())

- balance and income are numerical variables, with very different scales. We may need to standardise (especially if using regularisation).
- default and student are both objects and will need to be encoded.
- there are no missing values.

## Splitting and Preprocessing the Data

Next, let's create our feature matrix and response vector. We use `LabelEncoder` to encode our categorical output to a binary vector. 

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Feature matrix and response vector
X, y = df_default.drop(['default'], axis=1), df_default['default']

# Encode default
y = LabelEncoder().fit_transform(y)

print(X.shape)
print(y.shape)

# Print the class distribution before splitting the data set 
print(pd.Series(y).value_counts(normalize=True)*100)

Notice the high class imbalance, with 96.67% of individuals that have not defaulted on their loan. Consider the following naive train/test split.

In [ ]:
# Naively split the data into train and test sets 
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle= True,
                                                    test_size = 0.1, random_state=1112)


# Check the proportion of defaults in both train and test data sets 
print(pd.Series(y_train).value_counts(normalize=True)*100)
print(pd.Series(y_test).value_counts(normalize=True)*100)

### 🚩 Exercise 2 (CORE)

a. Why might you NOT want to use the train/test split above?

b. Modify the code to have similar class proportions in the train and test sets. Print the class proportions to check if they are similar. 

c. Visualise the training features and responses (balance, income, student) and comment on any differences that you observe between individuals that have defaulted.

<details><summary><b><u>Hint</b></u></summary>
Consider the the additional argument `stratify=` inside of the [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function to stratify in terms of the response when spliting the data.
</details>

a. Only 2% of individuals defaulted in the test dataset, while 3.48% defaulted in the training set. In absolute numbers, this difference may seem small, but the relative difference is quite large and can cause poor model performance. Indeed, the test data only contains 20 individuals who defaulted.

In [ ]:
print(pd.Series(y_test).value_counts())

In [ ]:
# Stratify split
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle= True, stratify=y,
                                                    test_size = 0.1, random_state=1112)


# Check the proportion of defaults in both train and test data sets 
print(pd.Series(y_train).value_counts(normalize=True)*100)
print(pd.Series(y_test).value_counts(normalize=True)*100)

In [ ]:
# Visualise the training data

fig, axes = plt.subplots(ncols = 2, nrows=2, figsize=(15,10))

sns.scatterplot(data = X_train, x = "balance", y="income", hue=y_train, ax=axes[0,0])
sns.scatterplot(data = X_train, x = "balance", y="income", hue="student", ax=axes[0,1])
sns.boxplot(data = X_train, x = y_train, y="balance", hue = "student", ax=axes[1,0])
sns.boxplot(data = X_train, x = y_train, y="income", hue = "student", ax=axes[1,1])
plt.tight_layout()
plt.show()


c. Visually, we observe that those who default tend to have a higher credit card balance, and students who default tend to have a higher balance. Students generally have lower income, but there appears to be little relationship between income and default.

# Logistic Regression <a id='RBH'></a>

Recall from our notes that for a binary output $y \in \lbrace 0, 1 \rbrace$, **logistic regression** is a classifier that can be seen a simple generalizsation of linear regression by making two changes. 

- First, we replaced the **Gaussian** distribution of the output $y$ with a **Bernoulli** distribution. 
- Second, we pass the linear function of the inputs, $\mathbf{w}^T\mathbf{x}$, through a **link function** $g: \mathbb{R} \rightarrow [0,1]$. 

That is, we assume $y \sim \text{Bern}( g(\mathbf{w}^T\mathbf{x}))$.

The link function takes values in the unit interval to ensure that the conditional probability of a success, 

$$p(y =1 \mid \mathbf{x}) = E[y | \mathbf{x}] = g(\mathbf{w}^T\mathbf{x})$$

is between zero and one. Specifically, in logistic regression, we select the **logistic** link function (S-shaped), defined as 

$$g(\mathbf{w}^T\mathbf{x}) = \frac{1}{1 + \exp(-\mathbf{w}^T\mathbf{x})}$$

Putting these steps together, the logistic regression model is:

$$y \sim \text{Bern}\left( \left[1 + \exp(-\mathbf{w}^T\mathbf{x})\right]^{-1} \right)$$

In `sklearn`, we can fit a logistic regression model using [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). 

**Creating a logistic regression model:** some important options include
- `l1_ratio` specifies the type of regularisation/penalty used: you can use l2 regularisation by setting `l1_ratio=0`, l1 regularisation by setting `l1_ratio=1`, and elastic net regularisation by setting `l1_ratio` to any float between 0 and 1.
- `C` is proportional to the inverse of the regularisation parameter, with `C=np.inf` resulting in no penalty.

**CAUTION:** by default `LogisticRegression` uses an l2 regularisation with penalty parameter `C=1`. This default should never be used! Instead, you should either use no penalty by setting `C=np.inf` or tune the value of `C` with cross-validation. 

**Fitting a logistic regression model:**
- use the usual `.fit(X,y)` to fit the model. The fitted object stores the intercept and coefficients in the usual attributes `.intercept_` and `.coef_`.

**Prediction:** methods include
- `predict` which predicts the class label (either 0 or 1), 
- `predict_proba` which predicts the class probabilities, and 
- `predict_log_proba` which predicts the log probabilities of each class.

Let's start by create a pipeline to for our logisitic regression model (with no penalty).

In [ ]:
# Pipeline
log_pipe = make_pipeline(
    ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    ),
    LogisticRegression(random_state=42, C=np.inf)
)

### 🚩 Exercise 3 (CORE)

- Fit the logistic regression model to the training data
- Run the code to plot the coefficients and comment on their interpretation.
- Compute the the accuracy score of the model on the testing data

<details><summary><b><u>Hint</b></u></summary>
    
You can use the `score` method of `LogisiticRegression` to compute the accuracy.
    
</details>


In [ ]:
# Fit the model using the training data
log_pipe.fit(X_train, y_train)

# accuracy on the test data
print('Accuracy on the test data:', log_pipe.score(X_test,y_test))

In [ ]:
# Plot logistic regression coefficients
w_df = get_coefs(log_pipe,plot=True, intercept=False, figtitle="Logisitic Regression")

In [ ]:
w_df

Interpreting the coefficients:
- if the individual is a student, the odds of defaulting on their loan decreases by a factor of $\exp(-0.55)$.
- if the credit balance is increased by one standard deviation, the odds of defaulting increases by a factor of $\exp(2.77)$.
- the effect of income is neglible.

## Evaluating Classification Models: Beyond Accuracy

Because of the imbalanced nature of the data, looking at accuracy alone is misleading. Indeed, a naive classifier that predicts no one defaults would achieve a high accuracy of 96.7%.  

A binary classifier can make two types of errors:

- Incorrectly assigning an individual __who defaults__ to the __no default__ category (FN)
- Incorrectly assigning an individual who __does not default__ to the __default__ category (FP)

To better understand the FN vs FP tradeoff, we can compute the confusion matrix. Recall from our notes, the quantities reported in the confusion matrix are:


$$\text{TP} = \sum_{n=1}^N \mathbb{I}(y_n=1)\mathbb{I}(\widehat{y}_n=1),\quad \text{FP} = \sum_{n=1}^N \mathbb{I}(y_n=0)\mathbb{I}(\widehat{y}_n=1)$$
$$\text{FN} = \sum_{n=1}^N \mathbb{I}(y_n=1)\mathbb{I}(\widehat{y}_n=0), \quad \text{TN} = \sum_{n=1}^N \mathbb{I}(y_n=0)\mathbb{I}(\widehat{y}_n=0)$$


where $y_n$ is the true class and $\widehat{y}_n$ is the estimated class. 

### 🚩 Exercise 4 (CORE)

1. Use [ConfusionMatrixDisplay.from_estimator](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator) to compute and visualise the confusion matrix. 
2. Comment on the importance of FNs compared to FPs to the credit card company.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(log_pipe, X_test, y_test)
plt.show()

From the perspective of the credit card company, **FN** are more important, as the company may have to take on the entire amount borrowed. Instead, **FP** represents a loss of profit, that is typically much less the finacial burden of FN. In this case, FN=19 and FP=6, and the company may prefer higher FP to reduce the number of FN.

Recall from our notes, the different evaluation measures can be defined from the confusion matrix, such as:

$$
\text{Accuracy} = \frac{\text{TP + TN}}{\text{TP + TN + FP + FN}}
$$

$$
\text{FPR} = \frac{\text{FP}}{\text{FP}+ \text{TN}}, \hspace{0.5cm} \text{Recall (TPR)} = \frac{\text{TP}}{\text{TP}+ \text{FN}} \hspace{0.5cm}
\text{Precision} = \frac{\text{TP}}{\text{TP}+ \text{FP}}
$$


$$
\text{F1-Score} = 2\left(\frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}\right) = \frac{\text{2TP}}{\text{2TP + FP + FN}}
$$

To compute these quantites and more, functions are available in `sklearn.metrics`, including:

1. [Recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) (True Positive Rate)
3. [Precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
4. [F1-score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
5. [AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

For a detailed list, see https://scikit-learn.org/1.5/modules/model_evaluation.html#classification-metrics.

### 🚩 Exercise 5 (CORE)

Compute the accuracy, recall, precision, and F1-score on the test data for the fitted model (using the functions in `sklearn.metrics`) and comment on the model's performance.

**Hint** you first need to use `.predict()` to predict the class labels.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

# Predict the class labels
y_test_pred = log_pipe.predict(X_test)

# Compute metrics
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("F1-score:", f1_score(y_test, y_test_pred))

Again, the model has a high accuracy, but this is misleading given the imbalanced nature of the data. In particular, the recall rate is quite low (and is less than 0.5 as FN>TP), which may be unacceptable for the company. Because of this, the F1-score is also low.

### 🚩 Exercise 6 (EXTRA)

Based on the confusion matrix computed below, use the equations above to derive:

1. False Positive Rate 
2. Recall
3. Precision
4. F1-score

without using any additional built-in function from any module. If computed correctly, the numbers should match the previous exercise.

In [ ]:
from sklearn.metrics import confusion_matrix

# Predict class labels
y_test_pred = log_pipe.predict(X_test)

# Compute confusion matrix
confmat = confusion_matrix(y_true = y_test, y_pred=y_test_pred)
confmat

In [ ]:
# Extracting the quantities from the confusion matrix 
FP = confmat[0,1]
TN = confmat[0,0]
TP = confmat[1,1]
FN = confmat[1,0]

#Calculation of FPR, Recall, Precision and F1 Score
# FPR
FPR = (FP)/(FP+TN)
# Recall
Recall = (TP)/(TP+FN)
# Precision
Precision = (TP)/(FP+TP)
# F1 Score
F1 = 2 * (Precision * Recall)/(Precision + Recall)

print('Solutions (rounded to three decimals):\n')
print('False Positive Rate (FPR): '+ '%.3f' % FPR)
print('Recall: '+ '%.3f' % Recall)
print('Precision: '+ '%.3f' % Precision)
print('F1 Score: '+ '%.3f' % F1)

### 🚩 Exercise 7 (CORE)

The handy function `RocCurveDisplay.from_estimator` is also available in `sklearn` for plotting  ROC curve for the fitted model. Use this function to plot the ROC curve and compute the AUC using `sklearn.metrics.roc_auc_score` on the test data. Comment on the model's performance based on these quantities.


<details><summary><b><u>Hint</b></u></summary>
You will need to use `predict_proba()` to compute the class probabilities for each data point, in order to compute the AUC.
</details>

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score

# Plot the ROC curve
fig, ax = plt.subplots(figsize=(6,6))
RocCurveDisplay.from_estimator(log_pipe, X_test, y_test,plot_chance_level=True, ax = ax)
plt.show()

# Compute the probabilities for each class on the test data (N_test by C matrix)
y_test_prob = log_pipe.predict_proba(X_test)

# Compute the AUC
print('AUC:', roc_auc_score(y_test, y_test_prob[:,1]))

Based on the ROC curve and AUC, the performance of the model appears very good. The ROC curve hugs the top left corner and the AUC is close to 1. However, as we know from lectures, the ROC and AUC can be misleading on imbalanced data.

### 🚩 Exercise 8 (CORE)

Now, use the function `PrecisionRecallDisplay.from_estimator` to plot the Precision-Recall curve for the fitted model on the test data. Comment on the model's performance based on this figure.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(6,6))
PrecisionRecallDisplay.from_estimator(log_pipe, X_test, y_test,plot_chance_level=True, ax = ax)
plt.show()

The curve is far from the ideal setting, i.e. it does not hug the top right corner, but it does perform better than chance. In contrast to the ROC curve, the precision-recall curve is more appropriate for this data set due to the class imbalance. It highlights that the model is not very precise when it identifies an individual that defaults.

# Regularisation <a id='SKV'></a>

Regularisation can be useful in logistic regression to deal with high-dimensional data and for variable selection. Recall that the parameter `C` represents the inverse strength of the penalty, and `l1_ratio` controls the balance between l1 (`l1_ratio=1`) and l2 regularisation (`l1_ratio=0`).

Note: the choice of the algorithm (aka solver) depends on the penalty chosen (see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for more details.)

As we saw last week, it is important to tune the penalty parameter `C` (and `l1_ratio` if using elastic net) when including regularisation. Before we start searching over hyperparameters, its worth noting that some of the folds may not have the same distribution of the classes. This is particulary important for imbalanced data and means the we could get a validation score that may be a poor estimate of performance (for example we may have a fold with very few positive classes or more than usual, which can cause large differences on imbalanced data). To address this,  when doing our grid search, we will use a `StratifiedKFold` to ensure the distribution of classes in our folds reflects the distribution in the larger data.

Run the code below to investigate the difference in class distributions across folds when using `Kfold` vs `StratifiedKfold`.


In [ ]:
# Consider 5 folds
KF = KFold(n_splits=5)
SKF = StratifiedKFold(n_splits=5)

fold_names = ["KFold", "StratifiedKFold"]
for i, K in enumerate([KF, SKF]):
    # Initialise an empty DataFrame to store counts for the current fold type
    fold_nos = pd.DataFrame()
    for j, (train_i, test_i) in enumerate(K.split(X_train, y_train)):
        # Compute value counts for the current fold and ensure it's a DataFrame with appropriate columns
        fold_no = pd.DataFrame(pd.Series(y_train[test_i]).value_counts()).T
        fold_no.index = ["Fold " + str(j)]  # Rename the index to reflect the fold number
        # Concatenate with the fold_nos DataFrame
        fold_nos = pd.concat([fold_nos, fold_no], axis=0)
    
    fold_nos.fillna(0, inplace=True)  # Fill missing values with 0 if any class was not present in a fold
    print(f"{fold_names[i]} counts per fold:\n", fold_nos)

Next, let's perform a grid search to tune the regularisation parameter with l2 regularisation. Notice that we have listed multiple metrics to save with the option `scoring=`. However, setting `refit=` will select the best model and refit according to the specfied metric.

In [ ]:
# This code snippet sets up and executes a grid search for tuning the penalty parameter
# of a logistic regression model with l2 regularisation using cross-validation. 

# Pipeline 
log_pipe_l2 = make_pipeline(
    ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    ),
    LogisticRegression(random_state=42, l1_ratio=0)
)

# Uncomment this line to find how the penalty parameter is called in the pipeline
#log_pipe.get_params()

# Possible C values: 
C_list = np.linspace(0.01, 10, num=151)


# Grid search CV:
log_rs = GridSearchCV(log_pipe_l2, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True),
                      refit = "accuracy", # Refits the best model on the entire dataset using the accuracy metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs.fit(X_train, y_train)

### 🚩 Exercise 9 (CORE)

a. Run the following code to plot the mean accuracy averaged across the validation folds (in black). Comment on the suggested value of `C`.

b. Choose a metric other than accuracy (i.e. f1, recall, or precision) and redraw the figure to plot the mean across the validation folds. Does your suggested value of `C` change?

In [ ]:
# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_accuracy and mean_test_accuracy columns
    regex = '(split[0-4]+|mean)_test_accuracy'
).assign(
    # Add the alphas as a column
    C = C_list
)

# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_accuracy'),
    var_name='fold',
    value_name='Accuracy'
)

# Plot the validation scores across folds
plt.figure(figsize=(7,5))
sns.lineplot(x='C', y='Accuracy', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='Accuracy', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

Looking at accuracy, the mean accuracy is fairly constant across `C` values, and any value except the smallest would be reasonable. We also notice that regularisation provides little improvement in terms of accuracy and little variability across folds (the scale of the y-axis is quite small).

In [ ]:
# Repeat for f1

# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_testf1 columns
    regex = '(split[0-4]+|mean)_test_f1'
).assign(
    # Add the alphas as a column
    C = C_list
)

# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_f1'),
    var_name='fold',
    value_name='F1'
)

# Plot the validation scores across folds
plt.figure(figsize=(7,5))
sns.lineplot(x='C', y='F1', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='F1', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

When looking at other metrics, such as  the F1 score (which is more appropriate for imbalanced data), our conclusions do not change. Regularisation seems to have little effect, and any value of `C` larger than 1 would be reasonable. We do observe more variability across folds. 

### 🚩 Exercise 10 (EXTRA)

a. Rebuild your model with `l1` regularisation and perform a grid search to tune the penalty parameter. Which metric have you chosen for refitting the model and why?

b. Plot the coefficients using the helper function. Which variables are included? How does the performance compare to the model with no regularisation?

In [ ]:
# Pipeline 
log_pipe_l1 = make_pipeline(
    ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    ),
    LogisticRegression(random_state=42, l1_ratio=1, solver="saga")
)

# Possible C values: 
C_list = np.linspace(0.01, 3, num=101)


# Grid search CV:
log_rs_l1 = GridSearchCV(log_pipe_l1, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True),
                      refit = "f1", # Refits the best model on the entire dataset using the f1 metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs_l1.fit(X_train, y_train)

In [ ]:
# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs_l1.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_testf1 columns
    regex = '(split[0-4]+|mean)_test_f1'
).assign(
    # Add the alphas as a column
    C = C_list
)

# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_f1'),
    var_name='fold',
    value_name='F1'
)

# Plot the validation scores across folds
plt.figure(figsize=(7,5))
sns.lineplot(x='C', y='F1', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='F1', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.axvline(log_rs_l1.best_params_['logisticregression__C'], 0,1, linestyle = '--', c = "grey")
plt.show()

In [ ]:
get_coefs(log_rs_l1.best_estimator_, plot=True, intercept = False)

In [ ]:
# Compute metrics on the test data
y_test_pred = log_rs_l1.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("F1-score:", f1_score(y_test, y_test_pred))

Due to the imbalanced nature of the data, we have chosen the F1 metric over accuracy to refit the model.

All variables are included in the model, and the performance on the test data is the same as when the model was fit with no regularisation. Again, we find that regularisation offers no improvement.

# Dealing with Imbalanced Data <a id='Imbal'></a>

We have already seen that when dealing with imbalanced data, we need to 
1. Consider a suitable performance metric both for testing and validation.
2. Use a suitable splitting strategy (stratified splitting) both in creating our test and validation sets.

But the majority class can still overwhelm the minority class when fitting the model. To investigate this, let's use the function  [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html), which reports summary of the precision, recall, F1 score for each class. Note that:
- the recall of the positive class is also known as *sensitivity*; recall of the negative class is *specificity*.
- it also includes the *macro average* (averaging the unweighted mean per label) as well as the weighted average (averaging the support-weighted mean per label). 

In [ ]:
# Compute and print the classification report for the model with no penalty
from sklearn.metrics import classification_report

print(classification_report(y_test, 
                            log_pipe.predict(X_test), 
                            target_names = ['No Default','Default']))

Notice, how the metrics are much higher for the majority class (individuals who have not deafulted). In this case, we may want to alter the training algorithm or data by:

1. Weighting the classes during training.
2. Resampling the data.

`LogisticRegression` offers the option to include class weights, and for example `class_weight="balanced"` will weight the classes by: $ N/(2 N_c) $, where  $ N_c $ for $c = 0, 1$ counts the number of observations in each class. Notice how the metrics in classification report change when including this option in the code below. 

In [ ]:
log_pipe_balan = make_pipeline(
    ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    ),
    LogisticRegression(random_state=42, C=np.inf, class_weight='balanced')
).fit(X_train, y_train)

print(classification_report(y_test, 
                            log_pipe_balan.predict(X_test), 
                            target_names = ['No Default','Default']))


In the following, the focus is on suitable **resampling** to change the distribution of the classes in our training data.

## Resampling

To alter the distribution of the classes in our training data, there are two main approaches:

- Under-sampling the majority class
- Over-sampling the minority class

We will be using Imbalanced-learn (imported as `imblearn`), an open source, MIT-licensed library relying on scikit-learn that provides tools for dealing with classification with imbalanced classes (see [here](https://imbalanced-learn.org/stable/introduction.html) for an introduction). Let's start by installing the `imblearn` package if necessary.

In [ ]:
# Install the imblearn if necessary - necessary on Noteable. If using your own local python installation, you may want to use conda or similar. 
!pip install imblearn

In the previous notebooks, we learned about _transformers_, which allow us to __alter the features__ and not the number of observations in our data (i.e. the columns of $\mathbf{X}$). Instead, _resamplers_ provide a preprocessing step in our `Pipeline` to __alter the number of observations__ and not the features (i.e. the rows of $\mathbf{X}$ and $\mathbf{y}$).

__Resamplers__

Resamplers are classes that follow the scikit-learn API and have a sampling functionality through the `.resample()` method. Like all other scikit-learn methods, they have a `.fit()` method which is only applied during pipeline training. This means if we want to create our own resampler from scratch that is compatible with scikit-learn, we just have to make a class that has three methods; `.fit()`,  `.resample()`, and `.fit_resample()`; with the latter just chaining the other two together.

Therefore to resample a dataset, each sampler implements:

```
obj.fit(data, targets)
data_resampled, targets_resampled = obj.resample(data, targets)
```

or simply...

```
data_resampled, targets_resampled = obj.fit_resample(data, targets)
```

**Remember to include your sampler within your model pipeline to prevent data leakage!**


### Under-Sampling

Under-sampling involves removing observations from the majority class to prevent its signal from dominating during training. We will focus on the simplest strategy:

- [`RandomUnderSampler`](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html), which randomly removes observations from the majority class.

For an overview of other under-samplers, see: https://imbalanced-learn.org/stable/under_sampling.html

In [ ]:
from imblearn.pipeline import Pipeline as ImPipeline
from imblearn.under_sampling import RandomUnderSampler

# Pipeline with under sampling
log_pipe_us = ImPipeline([
    ("fe",ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    )),
   ("sampler", RandomUnderSampler(random_state=42)),
   ("model", LogisticRegression(random_state=42, C=np.inf))])

# Fit model    
log_pipe_us.fit(X_train, y_train)

### 🚩 Exercise 11 (CORE)

Compute and print the classification report and visualize the confusion matrix for the model above with undersampling. How have the results changed?

In [ ]:
# Print classification report
print(classification_report(y_test, 
                            log_pipe_us.predict(X_test), 
                            target_names = ['No Default','Default']))

# Draw confusion matrix
ConfusionMatrixDisplay.from_estimator(log_pipe_us, X_test, y_test)
plt.show()

We notice that among the default cases, the precision and f1-score are worse but the recall is much higher. In particular, the model is better able to identify individuals that default but this comes at the cost of higher FP (not loaning to individuals who will not deafult on their loans). 

## Oversampling

Over-sampling involves generating additional observations from the minority class. We will focus on the simplest strategy:

- [`RandomOverSampler`](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html), which randomly samples with replacement observations from the minority class.

For an overview of other over-samplers, see: https://imbalanced-learn.org/stable/over_sampling.html

### 🚩 Exercise 12 (CORE)

a. Create a new pipeline for a logisitic regression model with oversampling (by copying and editting the pipeline above).

b. How do the performance metrics compare with under-sampling?

In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Pipeline with over sampling
log_pipe_os = ImPipeline([
    ("fe",ColumnTransformer(
        [("cat", OneHotEncoder(drop=["No"]), ['student']),
         ("num", StandardScaler(), ['balance','income'])],
         verbose_feature_names_out=False
    )),
   ("sampler", RandomOverSampler(random_state=42)),
   ("model", LogisticRegression(random_state=42, C=np.inf))])

# Fit the model    
log_pipe_os.fit(X_train, y_train)

In [ ]:
# Print the classification report
print(classification_report(y_test, 
                            log_pipe_os.predict(X_test), 
                            target_names = ['No Default','Default']))

# Visualize the confusion matrix
ConfusionMatrixDisplay.from_estimator(log_pipe_os, X_test, y_test)
plt.show()

The results are slightly better with over-sampling. The number of TP is increased from 29 to 30, and both the numbers of FP and FN are slightly lower. 

---
# Multi-class Logistic Regression <a id='mclog'></a>

Finally, to gain some experience with multi-class logistic regression, let's now look at the `iris` data set, which we have seen already in lectures.  

This is an example where the response has **3 classes** corresponding to the three types of iris species and $D=4$ features (petal length, petal width, sepal length, sepal width).

Let's start by loading the data and doing some basic EDA.

In [ ]:
# First load the data 
from sklearn.datasets import load_iris

# Loading data
iris = pd.DataFrame(sns.load_dataset('iris'))

# Print information about the data set
print(iris.info())
print(iris.describe())
print(iris['species'].value_counts())

From the intial EDA, we observe:
- The features have similar, but slightly different scales.
- The target variable `species` is a string and we will need a label encoding
- The species types are equal distributed, and thus no methods for class imbalance are needed.

Now, let's separate the features and target, encode the target, and split into training and test sets.

In [ ]:
# Feature matrix and response vector
X, y = iris.drop(['species'], axis=1), iris['species']

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# Split into training and testing 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True) 

Let's visualise the training data using a pairplot:

In [ ]:
# Pairplot
train_df  = pd.DataFrame(X_train)
train_df['species'] = y_train
sns.pairplot(data = train_df, hue = 'species',corner=True)
plt.show()

We further note that, visually, the species appear to be fairly well separated.

### 🚩 Exercise 13 (CORE)

We can use the same `LogisticRegression` for the multi-class setting. In this case, it will fit the **multinomial logistic regression** model. Fit a multinomial logisitic regression model and visualize the confusion matrix.

In [ ]:
# Multinomial logistic regression
lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=np.inf, random_state=42))
lr.fit(X_train,y_train)

# Plot the confusion matrix
ConfusionMatrixDisplay.from_estimator(lr, X_test, y_test)
plt.show()

### 🚩 Exercise 14 (EXTRA)

In the multi-class setting, the `.coeff_` attribute is a $ C \times D $ matrix of the model coefficents. Recall that the log odds of being in class $ c $ relative to class $ c' $ is:

$$ \log \left(\frac{p(y=c \mid \mathbf{x})}{p(y=c' \mid \mathbf{x})} \right) =  \log \left(\frac{\exp(\mathbf{w}_c^T\mathbf{x})}{\sum_{l=1}^C\exp(\mathbf{w}_l^T\mathbf{x})} /  \frac{\exp(\mathbf{w}_{c'}^T\mathbf{x})}{\sum_{l=1}^C\exp(\mathbf{w}_l^T\mathbf{x})} \right) = \left(\mathbf{w}_{c} - \mathbf{w}_{c'} \right)^T\mathbf{x}$$

For each unique pair of classes, create a barplot to visualize the coefficients of one class relative to another. 
Which features are most important for distinguishing between classes? Interpret how a unit change in the features affects the odds of being in one class relative to another.

In [ ]:
# Extract coefficients relative to a reference category
fe_names = lr[:-1].get_feature_names_out()
w0_ref2 = lr[-1].coef_[0,:]- lr[-1].coef_[2,:]
w1_ref2 = lr[-1].coef_[1,:]- lr[-1].coef_[2,:]
w0_ref1 = lr[-1].coef_[0,:]- lr[-1].coef_[1,:]

# Plot coefficients
fig, ax = plt.subplots(1,3,figsize=(12,4),sharey=True)
w02_df = pd.DataFrame({'feature': fe_names, 'coef': w0_ref2})
ax[0].barh(w02_df['feature'], w02_df['coef'])
ax[0].set_ylabel('Features')
ax[0].set_xlabel('Coefficient Value')
ax[0].axvline(x=0, color=".5")
ax[0].set_title(f'{le.classes_[0]} relative to {le.classes_[2]}')
w12_df = pd.DataFrame({'feature': fe_names, 'coef': w1_ref2})
ax[1].barh(w12_df['feature'], w12_df['coef'])
ax[1].set_ylabel('Features')
ax[1].set_xlabel('Coefficient Value')
ax[1].axvline(x=0, color=".5")
ax[1].set_title(f'{le.classes_[1]} relative to {le.classes_[2]}')
w01_df = pd.DataFrame({'feature': fe_names, 'coef': w0_ref1})
ax[2].barh(w01_df['feature'], w01_df['coef'])
ax[2].set_ylabel('Features')
ax[2].set_xlabel('Coefficient Value')
ax[2].axvline(x=0, color=".5")
ax[2].set_title(f'{le.classes_[0]} relative to {le.classes_[1]}')
plt.show()
    


There are three unique pairs of classes. 

- Setosa relative to virginica: the most important features are petal length and petal width. A unit increase in these features decreases the odds of flower being a setosa relative to virginica by a factor of approximately $\exp(-19)$ and  $\exp(-16)$, respectively.

- Versicolor relative to virginica: the most important features are again petal length and petal width. A unit increase in these features decreases the odds of flower being a versicolor relative to virginica by a factor of approximately $\exp(-14)$ and  $\exp(-12)$, respectively.

- Setosa relative to versicolor: the most important features is sepal length. A unit increase in this features decreases the odds of flower being a setosa relative to versicolor by a factor of approximately $\exp(-8)$.

### 🚩 Exercise 15 (CORE)

Have you used any form of generative AI when working on this assessment? If so, describe how you used which tool, and what you have learned from it.

_Type your answer here!_

# Completing the Worksheet

At this point you have hopefully been able to complete all the CORE exercises and attempted the EXTRA ones. Now is a good time to check the reproducibility of this document by restarting the notebook's kernel and rerunning all cells in order.

Before generating the PDF, please **change 'Student 1', 'Student 2' and 'Student 3' at the top of the notebook to include your name(s)**.

Once that is done and you are happy with everything, you can then run the following cell to generate your PDF. 

In [ ]:
!jupyter nbconvert --to pdf mlp_week06_solutions.ipynb 